# Pair-crop evidence on deployed UPT–YOLO matches

Attach MS COCO 2014 images and the v175 full prompt cache. CPU or one GPU. Enable Internet for CLIP. Eight reviewed VALID caption pairs are screened for unique detector ROI pairs.


In [ ]:
from pathlib import Path
import hashlib, json
from PIL import Image, ImageOps

COCO_ROOT = Path('/kaggle/input/datasets/vuthetam/mscoco-2014')
OUTPUT = Path('/kaggle/working/clip_pair_crop_detector_valid.json')
MODEL = 'openai/clip-vit-base-patch16'
REVISION = '57c216476eefef5ab752ec549e440a49ae4ae5f3'
AUDIT = [{'coco_id': 209028, 'image': 'COCO_val2014_000000209028.jpg', 'subject': 'person', 'object': 'skateboard', 'pair': ['a dog riding a skateboard with a person standing on a skateboard', 'a person riding a skateboard with a dog'], 'relation_pair': 'on / riding', 'selected': 'A', 'positive': 0}, {'coco_id': 499400, 'image': 'COCO_val2014_000000499400.jpg', 'subject': 'men', 'object': 'surfboards', 'pair': ['a group of men holding surfboards on a beach', 'a group of men standing next to two surfboards'], 'relation_pair': 'beside / holding', 'selected': 'B', 'positive': 0}, {'coco_id': 139781, 'image': 'COCO_val2014_000000139781.jpg', 'subject': 'woman', 'object': 'kite', 'pair': ['a woman holding a kite next to a bird', 'a woman flying a kite in front of a house'], 'relation_pair': 'flying / holding', 'selected': 'B', 'positive': 1}, {'coco_id': 432818, 'image': 'COCO_val2014_000000432818.jpg', 'subject': 'woman', 'object': 'kite', 'pair': ['a woman standing next to a kite on a beach', 'a woman flying a kite with a blue sky in the background'], 'relation_pair': 'beside / flying', 'selected': 'B', 'positive': 1}, {'coco_id': 456638, 'image': 'COCO_val2014_000000456638.jpg', 'subject': 'woman', 'object': 'suitcase', 'pair': ['a woman holding a suitcase next to a train', 'a woman standing next to a red suitcase'], 'relation_pair': 'beside / holding', 'selected': 'A', 'positive': 0}, {'coco_id': 214468, 'image': 'COCO_val2014_000000214468.jpg', 'subject': 'man', 'object': 'train', 'pair': ['a man standing on a train next to a forest', 'a man standing next to a red train'], 'relation_pair': 'beside / on', 'selected': 'B', 'positive': 0}, {'coco_id': 558661, 'image': 'COCO_val2014_000000558661.jpg', 'subject': 'woman', 'object': 'frisbee', 'pair': ['a woman throwing a frisbee in a park', 'a woman catching a frisbee in a park'], 'relation_pair': 'catching / throwing', 'selected': 'A', 'positive': 0}, {'coco_id': 295945, 'image': 'COCO_val2014_000000295945.jpg', 'subject': 'man', 'object': 'motorcycle', 'pair': ['a man sitting on a yellow motorcycle', 'a man standing next to a yellow motorcycle'], 'relation_pair': 'beside / on', 'selected': 'B', 'positive': 1}]

PROMPT_ROOT = Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
PROMPT_SHA = 'e90c79b3b04540630b4bd2d279a4070ef05da7f4b65970c952851c37e2423bf0'
files = list(PROMPT_ROOT.rglob('prompt_links.jsonl'))
if len(files) != 1:
    raise ValueError(f'Expected one prompt_links.jsonl under {PROMPT_ROOT}')
with files[0].open('rb') as stream:
    if hashlib.file_digest(stream, 'sha256').hexdigest() != PROMPT_SHA:
        raise ValueError('Unexpected v175 prompt cache payload')

def detector_pair(example, row):
    if row is None or row['filename'] != example['image']:
        raise ValueError(f"Missing or mismatched VALID cache row: {example['coco_id']}")
    target = {'surfboards': 'surfboard'}.get(example['object'], example['object'])
    objects = {item['instance_id']: item for item in row['objects']}
    matches = {}
    for relation in row['relations']:
        subject = objects[relation['subject_instance']]
        obj = objects[relation['object_instance']]
        if subject['label'] != 'person' or obj['label'] != target:
            continue
        key = (relation['subject_roi'], relation['object_roi'])
        if key != (subject['roi'], obj['roi']):
            raise ValueError('Cache ROI identity mismatch')
        boxes = (subject['roi_bbox'], obj['roi_bbox'])
        if key in matches and matches[key] != boxes:
            raise ValueError('Conflicting boxes for one ROI pair')
        matches[key] = boxes
    if len(matches) != 1:
        return None, len(matches)
    return next(iter(matches.values())), 1

wanted = {item['coco_id'] for item in AUDIT}
links = {}
with files[0].open(encoding='utf-8') as stream:
    for line in stream:
        row = json.loads(line)
        if row['group'] == 'val' and row['coco_id'] in wanted:
            if row['coco_id'] in links:
                raise ValueError('Duplicate VALID cache row')
            links[row['coco_id']] = row

eligible = []
excluded = []
for example in AUDIT:
    boxes, count = detector_pair(example, links.get(example['coco_id']))
    if boxes is None:
        excluded.append(dict(coco_id=example['coco_id'], matching_roi_pairs=count,
                             reason='no_pair' if count == 0 else 'ambiguous_pair'))
    else:
        eligible.append(dict(example, roi_boxes=boxes))
print(f'{len(eligible)}/{len(AUDIT)} reviewed pairs with a unique person-object ROI pair')
print(json.dumps(excluded, indent=2))


In [ ]:
"""Pixel-exact crop rectangle and original-image patch coordinates; no model imports."""
import math


def crop_rectangle(subject,obj,width,height):
    assert width>0 and height>0
    for b in (subject,obj):assert len(b)==4 and 0<=b[0]<b[2]<=1 and 0<=b[1]<b[3]<=1
    x0=min(subject[0],obj[0])*width;y0=min(subject[1],obj[1])*height
    x1=max(subject[2],obj[2])*width;y1=max(subject[3],obj[3])*height
    dx=.1*(x1-x0);dy=.1*(y1-y0)
    return [max(0,math.floor(x0-dx)),max(0,math.floor(y0-dy)),min(width,math.ceil(x1+dx)),min(height,math.ceil(y1+dy))]




rows = []
if eligible:
    import torch
    from transformers import CLIPModel, CLIPProcessor

    names = {item['image'] for item in eligible}
    paths = {}
    for path in COCO_ROOT.rglob('COCO_*.jpg'):
        if path.name in names:
            if path.name in paths:
                raise ValueError(f'Duplicate image: {path.name}')
            paths[path.name] = path
    if paths.keys() != names:
        raise FileNotFoundError(f'Missing images: {sorted(names - paths.keys())}')
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    processor = CLIPProcessor.from_pretrained(MODEL, revision=REVISION)
    model = CLIPModel.from_pretrained(MODEL, revision=REVISION, use_safetensors=False).to(device).eval()
    with torch.inference_mode():
        for item in eligible:
            path = paths[item['image']]
            with Image.open(path) as original:
                image = original.convert('RGB')
            rect = crop_rectangle(*item['roi_boxes'], *image.size)
            views = [ImageOps.pad(view, (224, 224), method=Image.Resampling.BICUBIC,
                                  color=(123, 117, 104)) for view in (image, image.crop(rect))]
            batch = processor(text=item['pair'], images=views, padding=True, return_tensors='pt')
            result = model(**batch.to(device))
            img = torch.nn.functional.normalize(result.image_embeds.float())
            txt = torch.nn.functional.normalize(result.text_embeds.float())
            scores = (img @ txt.T).cpu().tolist()
            if not all(math.isfinite(v) for pair in scores for v in pair):
                raise ValueError(f'Nonfinite CLIP score: {path.name}')
            idx = item['positive']
            margins = dict(zip(('full', 'crop'), [pair[idx]-pair[1-idx] for pair in scores], strict=True))
            rows.append(dict(item, image_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                             crop_xyxy=rect, similarities=scores, margins=margins))
else:
    device = None

eps = 1e-6
summary = dict(eligible=len(rows), screened=len(AUDIT))
for mode in ('full', 'crop'):
    margins = [row['margins'][mode] for row in rows]
    summary[mode] = dict(correct=sum(v > eps for v in margins), tied=sum(abs(v) <= eps for v in margins),
                         mean_margin=sum(margins)/len(margins) if margins else None)
summary['wrong_to_right'] = sum(r['margins']['full'] < -eps and r['margins']['crop'] > eps for r in rows)
summary['right_to_wrong'] = sum(r['margins']['full'] > eps and r['margins']['crop'] < -eps for r in rows)
payload = dict(model=MODEL, revision=REVISION, device=device, split='val', prompt_links_sha256=PROMPT_SHA,
               summary=summary, excluded=excluded, records=rows)
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
OUTPUT.write_text(json.dumps(payload, indent=2, allow_nan=False), encoding='utf-8')
print(json.dumps(summary, indent=2))
print('SEND:', OUTPUT)
